In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
x = np.array([
    100, 150, 200,
    400, 500, 600,
    750, 850, 950
], dtype=float)

y = np.array([
    0, 0, 0,       # Dark
    1, 1, 1,       # Normal
    2, 2, 2        # Bright
])

In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=[1]),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [5]:
model.fit(x, y, epochs=500, verbose=0)

print("Training completed!")

Training completed!


In [6]:
test_x = np.array([150, 500, 900], dtype=float)

predictions = model.predict(test_x)

categories = ["Dark", "Normal", "Bright"]

for i in range(len(test_x)):
    predicted_class = np.argmax(predictions[i])
    print(
        "Light Level =", test_x[i],
        "Original Model =", categories[predicted_class]
    )

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Light Level = 150.0 Original Model = Dark
Light Level = 500.0 Original Model = Bright
Light Level = 900.0 Original Model = Bright


In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("light_classifier.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

Saved artifact at '/tmp/tmpdcqd260m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133570577150032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133570577150992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133570577150608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133570577151376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133570577151184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133570577151760: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved successfully!


In [9]:
interpreter = tf.lite.Interpreter(
    model_path="light_classifier.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite model loaded successfully!")

TFLite model loaded successfully!


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [10]:
for value in test_x:
    input_data = np.array([[value]], dtype=np.float32)

    interpreter.set_tensor(
        input_details[0]['index'],
        input_data
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]['index']
    )

    predicted_class = np.argmax(output[0])

    print(
        "Light Level =", value,
        "TFLite Model =", categories[predicted_class]
    )

Light Level = 150.0 TFLite Model = Dark
Light Level = 500.0 TFLite Model = Bright
Light Level = 900.0 TFLite Model = Bright
